# Indian-law injection-benchmark corpus — downloader

Fetches every dataset that supplies **(context, question, gold answer)** triples for
Indian law / Indian financial regulation, and lays them out under `datasets/`.

| # | Dataset | Rows | Context | License |
|---|---------|------|---------|---------|
| 1 | IndiaFinBench | 406 | inline | CC BY 4.0 |
| 2 | BNS / BNSS / BSA QA | 6,354 | join on section | Apache 2.0 |
| 3 | IndicLegalQA | 10,000 | none | CC BY 4.0 |
| 4 | CourtNav RAG (adalat-ai) | 21 | inline + PDFs | Apache 2.0 |
| 5 | Law entrance exams (adalat-ai) | 6,218 | MCQ, passage inline | MIT |
| 6 | IL-TUR | 484,316 | no question field | CC BY-NC-SA 4.0 |

Every cell is **idempotent** — an existing, correctly-sized file is skipped.
Run cells top to bottom.

## 1 · Setup

Reads `HF_TOKEN` from `.env`. Datasets 4 and 5 are gated (`gated: "auto"` — auto-approved,
but the request must be authenticated), so the token is required for those two.

In [1]:
import hashlib, json, os, shutil, sys
import urllib.request, urllib.parse, urllib.error
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "datasets"
DATA.mkdir(exist_ok=True)

# --- load .env -------------------------------------------------------------
def load_env(path=ROOT / ".env"):
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

ENV = load_env()
HF_TOKEN = ENV.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
HF_HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

print(f"HF_TOKEN: {'loaded (' + str(len(HF_TOKEN)) + ' chars)' if HF_TOKEN else 'MISSING — gated datasets 4 & 5 will fail'}")
print(f"root:     {ROOT}")

# --- disk check ------------------------------------------------------------
free = shutil.disk_usage(ROOT).free
print(f"free:     {free / 1e9:.2f} GB")
if free < 2e9:
    print("  ! low disk — IL-TUR (1.6 GB, cell 7) will not fit. Everything else needs ~20 MB.")

HF_TOKEN: loaded (37 chars)
root:     /Users/mihirmohite/BeyondBot/Luna Benchmark Reasoning
free:     10.72 GB


In [2]:
def human(n):
    for u in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.0f}{u}" if u == "B" else f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"


# Mendeley 403s the default "Python-urllib/3.x" agent, so always send a browser UA.
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36"}


def fetch(url, dest, headers=None, force=False, min_bytes=1024):
    """Download url -> dest. Skips if dest already exists and is non-trivial."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= min_bytes and not force:
        print(f"  skip  {dest.relative_to(ROOT)}  ({human(dest.stat().st_size)}, already present)")
        return dest
    req = urllib.request.Request(url, headers={**UA, **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=300) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    except Exception as e:
        code = getattr(e, "code", type(e).__name__)
        print(f"  FAIL  {dest.name}  [{code}]  {url}")
        if dest.exists():
            dest.unlink()
        return None
    print(f"  ok    {dest.relative_to(ROOT)}  ({human(dest.stat().st_size)})")
    return dest


def sha256(path, n=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(n):
            h.update(chunk)
    return h.hexdigest()


print("helpers ready")

helpers ready


## 2 · Verify the two files already downloaded manually, then fold them in

`test.csv` landed at a mirrored URL path (`adalat-ai/.../resolve/main/`) and the CLAT parquet
landed in the project root. Both are checked once here, then moved into the layout and not
re-downloaded.

Note only **one of three** exam splits was fetched — `clat_pg` and `djs_dhjs` are pulled in cell 6.

In [3]:
# --- migrate any files saved under earlier names, so nothing is downloaded twice ---
LEGACY = {
    "indiafinbench/indiafinbench_qa_combined_406.json": "indiafinbench/indiafinbench_qa.json",
    "indiafinbench/indiafinbench_v1.csv":               "indiafinbench/indiafinbench_qa.csv",
    "bns_bnss_bsa/bns_legal_qa.jsonl":                  "bns_bnss_bsa/bns_qa.jsonl",
    "bns_bnss_bsa/bnss_legal_qa.jsonl":                 "bns_bnss_bsa/bnss_qa.jsonl",
    "bns_bnss_bsa/bsa_legal_qa.jsonl":                  "bns_bnss_bsa/bsa_qa.jsonl",
    "indiclegalqa/IndicLegalQA_10K_Revised.json":       "indiclegalqa/indiclegalqa_revised.json",
    "indiclegalqa/IndicLegalQA_10K.json":               "indiclegalqa/indiclegalqa_original.json",
}
for old, new in LEGACY.items():
    o, n = DATA / old, DATA / new
    if not o.exists():
        continue
    if n.exists():
        o.unlink()
        print(f"dedup: removed {old} ({new} already present)")
    else:
        n.parent.mkdir(parents=True, exist_ok=True)
        o.rename(n)
        print(f"renamed: {old} -> {new}")

STRAYS = [
    # (current path, destination, expected rows, expected columns)
    (ROOT / "adalat-ai/Indian-Legal-Retrieval-Generation/resolve/main/test.csv",
     DATA / "courtnav_rag/courtnav_rag_test.csv",
     21, ["Query", "Context", "Document", "Gold Answers"]),
    (ROOT / "clat_ug-00000-of-00001.parquet",
     DATA / "law_entrance_exams/clat_ug.parquet",
     3154, ["question_text", "options", "answer", "source_paper"]),
]

for src, dest, n_rows, cols in STRAYS:
    if dest.exists():
        print(f"already placed: {dest.relative_to(ROOT)}")
        continue
    if not src.exists():
        print(f"not found (will download later): {src.name}")
        continue
    df = pd.read_csv(src) if src.suffix == ".csv" else pd.read_parquet(src)
    ok = (len(df) == n_rows) and (list(df.columns) == cols)
    print(f"{src.name}: {df.shape} cols={list(df.columns)} -> {'OK' if ok else 'MISMATCH'}")
    if not ok:
        print(f"  expected {n_rows} rows / {cols} — NOT moving, inspect manually")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest))
    print(f"  moved -> {dest.relative_to(ROOT)}")

# remove the now-empty mirrored URL directory
stale = ROOT / "adalat-ai"
if stale.exists() and not any(p.is_file() for p in stale.rglob("*")):
    shutil.rmtree(stale)
    print("removed empty ./adalat-ai/ URL-mirror tree")

already placed: datasets/courtnav_rag/courtnav_rag_test.csv
already placed: datasets/law_entrance_exams/clat_ug.parquet


## 3 · IndiaFinBench — 406 items, the only set that natively ships `context`

`Rajveer-code/IndiaFinBench` · CC BY 4.0 · arXiv 2604.19298 (under review, FinNLP @ EMNLP 2026).

**Two record schemas** — a loader that assumes one will crash:

* **A** (344 rows) — `context`, `document`, *optional* `regulation`
* **B** (62 rows, all `contradiction_detection`) — `context_a`/`context_b`,
  `document_a`/`document_b`, *optional* `regulation_a`/`regulation_b`, plus `explanation`

`regulation` is absent on 84 rows.

In [4]:
IFB = "https://raw.githubusercontent.com/Rajveer-code/IndiaFinBench/main"
d = DATA / "indiafinbench"

print("IndiaFinBench:")
fetch(f"{IFB}/annotation/raw_qa/indiafinbench_qa_combined_406.json", d / "indiafinbench_qa.json")
fetch(f"{IFB}/data/benchmark/indiafinbench_v1.csv",                  d / "indiafinbench_qa.csv")
fetch(f"{IFB}/evaluation/results_judged/judge_audit_log.csv",        d / "judge_audit_log.csv")

IndiaFinBench:
  skip  datasets/indiafinbench/indiafinbench_qa.json  (481.8KB, already present)
  skip  datasets/indiafinbench/indiafinbench_qa.csv  (298.6KB, already present)
  skip  datasets/indiafinbench/judge_audit_log.csv  (369.7KB, already present)


PosixPath('/Users/mihirmohite/BeyondBot/Luna Benchmark Reasoning/datasets/indiafinbench/judge_audit_log.csv')

In [5]:
rows = json.loads((DATA / "indiafinbench/indiafinbench_qa.json").read_text())
single = [r for r in rows if "context"   in r]
dual   = [r for r in rows if "context_a" in r]

print(f"total {len(rows)}  |  schema A (single-context) {len(single)}  |  schema B (dual-context) {len(dual)}")
print("task_type:  ", dict(Counter(r["task_type"]   for r in rows)))
print("answer_type:", dict(Counter(r["answer_type"] for r in rows)))
print("difficulty: ", dict(Counter(r["difficulty"]  for r in rows)))
print("source:     ", dict(Counter(r["source"]      for r in rows)))
print("missing 'regulation':", sum(1 for r in rows if "regulation" not in r and "regulation_a" not in r))

L = [len(r["context"]) for r in single]
print(f"context chars: min {min(L)} / median {sorted(L)[len(L)//2]} / max {max(L)}")

total 406  |  schema A (single-context) 344  |  schema B (dual-context) 62
task_type:   {'regulatory_interpretation': 174, 'numerical_reasoning': 92, 'contradiction_detection': 62, 'temporal_reasoning': 78}
answer_type: {'extractive': 267, 'calculated': 77, 'yes_no': 62}
difficulty:  {'easy': 160, 'medium': 182, 'hard': 64}
source:      {'SEBI': 338, 'RBI': 68}
missing 'regulation': 84
context chars: min 117 / median 382 / max 988


## 4 · BNS / BNSS / BSA — 6,354 QA over the 2023 criminal codes

`GSMS-B/Indian-Legal-QA-BNS-BNSS-BSA` · Apache 2.0 · public, no token needed.

Replaced the IPC, CrPC, and Evidence Act in July 2024 — the only set here on **current** law.
Template-generated: exactly 6 questions per section (`definitional_topic`, `definitional_section`,
`scenario`, `elements`, `exceptions`, `consequence`). 1,059 sections x 6 = 6,354.

**Two defects handled below:**
1. Row `BNSS_158` uses key `sectionnumber` instead of `section_number` — one row out of 6,354,
   and it breaks the HuggingFace dataset viewer for the whole repo (`CastError`).
2. `bsa_legal_qa.jsonl` carries a UTF-8 BOM — needs `encoding="utf-8-sig"`.

Raw files are downloaded verbatim; a repaired copy is written alongside them.

In [6]:
BNS_BASE = "https://huggingface.co/datasets/GSMS-B/Indian-Legal-QA-BNS-BNSS-BSA/resolve/main"
d = DATA / "bns_bnss_bsa"

print("BNS / BNSS / BSA (raw):")
for remote, local in [("bns_legal_qa",  "bns_qa.jsonl"),
                      ("bnss_legal_qa", "bnss_qa.jsonl"),
                      ("bsa_legal_qa",  "bsa_qa.jsonl")]:
    fetch(f"{BNS_BASE}/{remote}.jsonl", d / local)

BNS / BNSS / BSA (raw):
  skip  datasets/bns_bnss_bsa/bns_qa.jsonl  (1.1MB, already present)
  skip  datasets/bns_bnss_bsa/bnss_qa.jsonl  (1.7MB, already present)
  skip  datasets/bns_bnss_bsa/bsa_qa.jsonl  (504.2KB, already present)


In [7]:
d = DATA / "bns_bnss_bsa"
repaired, n_fixed = [], 0

for f in ["bns_qa.jsonl", "bnss_qa.jsonl", "bsa_qa.jsonl"]:
    # utf-8-sig transparently strips the BOM if present, and is a no-op if not
    with open(d / f, encoding="utf-8-sig") as fh:
        rows = [json.loads(line) for line in fh if line.strip()]
    for r in rows:
        if "sectionnumber" in r:                       # defect 1
            r["section_number"] = r.pop("sectionnumber")
            n_fixed += 1
    keys = Counter(tuple(sorted(r.keys())) for r in rows)
    print(f"{f:16s} {len(rows):5d} rows | keysets: {len(keys)} | "
          f"types: {dict(Counter(r['question_type'] for r in rows))}")
    repaired.extend(rows)

out = d / "bns_bnss_bsa_combined_clean.jsonl"
with open(out, "w", encoding="utf-8") as fh:
    for r in repaired:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\nrepaired {n_fixed} malformed key(s); wrote {len(repaired)} rows -> {out.relative_to(ROOT)}")
print("sections:", len({(r["act"], r["section_number"]) for r in repaired}))

bns_qa.jsonl      2148 rows | keysets: 1 | types: {'definitional_topic': 358, 'definitional_section': 358, 'scenario': 358, 'elements': 358, 'exceptions': 358, 'consequence': 358}
bnss_qa.jsonl     3186 rows | keysets: 1 | types: {'definitional_topic': 531, 'definitional_section': 531, 'scenario': 531, 'elements': 531, 'exceptions': 531, 'consequence': 531}
bsa_qa.jsonl      1020 rows | keysets: 1 | types: {'definitional_topic': 170, 'definitional_section': 170, 'scenario': 170, 'elements': 170, 'exceptions': 170, 'consequence': 170}

repaired 1 malformed key(s); wrote 6354 rows -> datasets/bns_bnss_bsa/bns_bnss_bsa_combined_clean.jsonl
sections: 1059


## 5 · IndicLegalQA — 10,000 QA from 1,253 Supreme Court judgments

Mendeley DOI `10.17632/gf8n8cnmvc.2` · CC BY 4.0 · **GPT-4o-generated**, manually reviewed.

No context field and no usable pointer to one — recovering context means joining case names
against IndianKanoon.

The two published files are **not the same dataset**: 9,100 questions shared, 859 only in the
original, 882 only in the revised. The original also has spelling drift
(`judgment_date` vs `judgement_date`) and 608 rows carrying a `reference_pdf` field of opaque
codes (90 distinct, e.g. `civ6`) with no shipped mapping. **Use the revised file.**

In [8]:
MENDELEY = "https://data.mendeley.com/public-files/datasets/gf8n8cnmvc/files"
d = DATA / "indiclegalqa"

print("IndicLegalQA:")
fetch(f"{MENDELEY}/8b19bfe1-8376-4258-8746-aec1c7fe11fd/file_downloaded", d / "indiclegalqa_revised.json")
fetch(f"{MENDELEY}/61830ed2-6e68-48a1-ab25-d8d257f82f02/file_downloaded", d / "indiclegalqa_original.json")

IndicLegalQA:
  skip  datasets/indiclegalqa/indiclegalqa_revised.json  (4.8MB, already present)
  skip  datasets/indiclegalqa/indiclegalqa_original.json  (4.7MB, already present)


PosixPath('/Users/mihirmohite/BeyondBot/Luna Benchmark Reasoning/datasets/indiclegalqa/indiclegalqa_original.json')

In [9]:
pr, po = DATA / "indiclegalqa/indiclegalqa_revised.json", DATA / "indiclegalqa/indiclegalqa_original.json"
if not (pr.exists() and po.exists()):
    print("IndicLegalQA files missing — re-run the download cell above.")
    raise SystemExit

rev  = json.loads(pr.read_text())
orig = json.loads(po.read_text())

print(f"revised  {len(rev):6d} rows | keysets: {Counter(tuple(r.keys()) for r in rev).most_common()}")
print(f"original {len(orig):6d} rows | keysets: {len(Counter(tuple(r.keys()) for r in orig))} variants")

sr = {(r['case_name'], r['question']) for r in rev}
so = {(r['case_name'], r['question']) for r in orig}
print(f"shared {len(sr & so)} | only-revised {len(sr - so)} | only-original {len(so - sr)}")
print(f"revised: {len({r['case_name'] for r in rev})} distinct cases")

revised   10000 rows | keysets: [(('case_name', 'judgement_date', 'question', 'answer'), 10000)]
original  10002 rows | keysets: 4 variants
shared 9100 | only-revised 882 | only-original 859
revised: 1253 distinct cases


## 6 · adalat-ai (gated — needs `HF_TOKEN`)

Both repos are `gated: "auto"`: auto-approved, but anonymous `resolve` returns **HTTP 401**.

**CourtNav RAG** (`Indian-Legal-Retrieval-Generation`, Apache 2.0, arXiv 2601.05255) — 21
lawyer-verified rows. The gold standard for record shape: `Context` holds numbered spans with
page references (`[1] Doc 1 ... - Page 1: ...`) and `Gold Answers` carries inline citation
markers pointing back at them, so answer correctness and citation grounding score separately.
The 4 source PDFs ship with it.

**Law entrance exams** (`indian-legal-exam-benchmark`, MIT, arXiv 2510.17900) — 6,218 MCQs.
`clat_ug` was fetched manually; `clat_pg` and `djs_dhjs` are pulled here.

In [10]:
RG   = "https://huggingface.co/datasets/adalat-ai/Indian-Legal-Retrieval-Generation/resolve/main"
EXAM = "https://huggingface.co/datasets/adalat-ai/indian-legal-exam-benchmark/resolve/main"

if not HF_TOKEN:
    print("HF_TOKEN missing — skipping. Add HF_TOKEN=... to .env and re-run this cell.")
else:
    print("CourtNav RAG:")
    fetch(f"{RG}/test.csv", DATA / "courtnav_rag/courtnav_rag_test.csv", HF_HEADERS)

    pdfs = {
        "Doc 1 (special power of attorney).pdf": "doc1_special_power_of_attorney.pdf",
        "Doc 2 - Indian Contract Act.pdf":       "doc2_indian_contract_act.pdf",
        "Doc 3 - DRT Application (1).pdf":       "doc3_drt_application.pdf",
        "Doc 4 - Civil Revision Petition.pdf":   "doc4_civil_revision_petition.pdf",
    }
    for remote, local in pdfs.items():
        url = f"{RG}/court_nav_small_data/{urllib.parse.quote(remote)}"
        fetch(url, DATA / "courtnav_rag/source_docs" / local, HF_HEADERS)

    print("\nLaw entrance exams:")
    for split in ["clat_ug", "clat_pg", "djs_dhjs"]:
        fetch(f"{EXAM}/data/{split}-00000-of-00001.parquet",
              DATA / "law_entrance_exams" / f"{split}.parquet", HF_HEADERS)

CourtNav RAG:
  skip  datasets/courtnav_rag/courtnav_rag_test.csv  (66.2KB, already present)
  skip  datasets/courtnav_rag/source_docs/doc1_special_power_of_attorney.pdf  (44.1KB, already present)
  skip  datasets/courtnav_rag/source_docs/doc2_indian_contract_act.pdf  (554.8KB, already present)
  skip  datasets/courtnav_rag/source_docs/doc3_drt_application.pdf  (236.4KB, already present)
  skip  datasets/courtnav_rag/source_docs/doc4_civil_revision_petition.pdf  (118.9KB, already present)

Law entrance exams:
  skip  datasets/law_entrance_exams/clat_ug.parquet  (953.9KB, already present)
  skip  datasets/law_entrance_exams/clat_pg.parquet  (176.8KB, already present)
  skip  datasets/law_entrance_exams/djs_dhjs.parquet  (451.6KB, already present)


In [11]:
p = DATA / "courtnav_rag/courtnav_rag_test.csv"
if p.exists():
    df = pd.read_csv(p)
    print(f"courtnav_rag_test.csv  {df.shape}  {list(df.columns)}")
    r = df.iloc[0]
    print(f"  Query:       {r['Query'].strip()[:100]}")
    print(f"  Context:     {r['Context'][:150].strip()} ...")
    print(f"  Gold Answer: {r['Gold Answers'][:180]}")

print()
for split, n in [("clat_ug", 3154), ("clat_pg", 814), ("djs_dhjs", 2250)]:
    q = DATA / "law_entrance_exams" / f"{split}.parquet"
    if q.exists():
        df = pd.read_parquet(q)
        flag = "OK" if len(df) == n else f"EXPECTED {n}"
        print(f"{split:10s} {len(df):5d} rows  papers={df['source_paper'].nunique():3d}  {flag}")
    else:
        print(f"{split:10s} missing")

courtnav_rag_test.csv  (21, 4)  ['Query', 'Context', 'Document', 'Gold Answers']
  Query:       Where is the property situated?
  Context:     [1] Doc 1 (special power of attorney).pdf – Page 1: We, the Principals, are the absolute owners of the residential property (house) bearing present No ...
  Gold Answer: The property is situated at Present No.62/A, Old No.103/1, consisting of building and vacant land on IV Main Road (Formerly Third Road), Gavipuram Extension, 31st Division, Bangalo

clat_ug     3154 rows  papers= 18  OK
clat_pg      814 rows  papers=  7  OK
djs_dhjs    2250 rows  papers= 13  OK


## 7 · IL-TUR — 1.6 GB, 8 configs / 36 splits, and probably not what you want

`Exploration-Lab/IL-TUR` · **CC BY-NC-SA 4.0** (non-commercial, share-alike) · gated via
`extra_gated_fields` (Full Name, Affiliation …), so access must be granted on your account.

**It has no question field.** All 8 tasks are `document -> label`: NER spans, per-sentence
rhetorical roles, binary GRANTED/DENIED, binary appeal outcome, multi-label IPC sections,
ranked doc IDs. Only `SUMM` and `L_MT` have free-text targets, and those are summarization and
translation. It cannot serve an injection benchmark without synthesising questions.

Also: its `LSI` task is built on **100 IPC sections** — repealed in July 2024.

Config names differ from the paper's task labels — the real ones are
`bail, cjpe, lmt, lner, lsi, pcr, rr, summ` (note `lmt`, not `l_mt`):

| config | splits |
|---|---|
| `bail` | train/dev/test × `_all`, `_specific` (6) |
| `cjpe` | `expert`, `single_train/dev`, `multi_train/dev`, `test` (6) |
| `lmt` | `acts`, `cci_faq`, `ip` (3) |
| `lner` | `fold_1/2/3` (3) |
| `lsi` | `train`, `dev`, `test`, `statutes` (4) |
| `pcr` | train/dev/test × `_candidates`, `_queries` (6) |
| `rr` | `CL_` and `IT_` × train/dev/test (6) |
| `summ` | `train`, `test` (2) |

In [12]:
WANT_ILTUR = True           # <- set False to skip (~1.6 GB across 39 parquet files)

# The gate hides the repo file tree even from an authorised token (`/api/.../tree` returns 0
# blobs on both `main` and `script`). The datasets-server auto-converted parquet branch
# (`refs/convert/parquet`) *does* resolve with the token, so pull from there instead of
# load_dataset — which would try to read the hidden tree and fail.
ILTUR_PARQUET_API = "https://datasets-server.huggingface.co/parquet?dataset=Exploration-Lab%2FIL-TUR"

if not WANT_ILTUR:
    print("skipped (WANT_ILTUR = False)")
elif not HF_TOKEN:
    print("HF_TOKEN missing — IL-TUR is gated, skipping.")
elif shutil.disk_usage(ROOT).free < 3e9:
    print(f"refusing: {shutil.disk_usage(ROOT).free/1e9:.2f} GB free, need ~3 GB headroom for 1.6 GB")
else:
    req = urllib.request.Request(ILTUR_PARQUET_API, headers={**UA, **HF_HEADERS})
    with urllib.request.urlopen(req, timeout=180) as r:
        files = json.load(r)["parquet_files"]

    total = sum(f["size"] for f in files)
    print(f"{len(files)} parquet files, {total/1e6:.1f} MB across "
          f"{len({f['config'] for f in files})} configs\n")

    d = DATA / "il_tur"
    for f in sorted(files, key=lambda x: (x["config"], x["split"], x["filename"])):
        dest = d / f["config"] / f["split"] / f["filename"]
        fetch(f["url"], dest, HF_HEADERS, min_bytes=f["size"])

39 parquet files, 1598.7 MB across 8 configs

  skip  datasets/il_tur/bail/dev_all/0000.parquet  (36.2MB, already present)
  skip  datasets/il_tur/bail/dev_specific/0000.parquet  (31.4MB, already present)
  skip  datasets/il_tur/bail/test_all/0000.parquet  (71.4MB, already present)
  skip  datasets/il_tur/bail/test_specific/0000.parquet  (62.5MB, already present)
  skip  datasets/il_tur/bail/train_all/0000.parquet  (155.9MB, already present)
  skip  datasets/il_tur/bail/train_all/0001.parquet  (94.5MB, already present)
  skip  datasets/il_tur/bail/train_specific/0000.parquet  (134.4MB, already present)
  skip  datasets/il_tur/bail/train_specific/0001.parquet  (77.2MB, already present)
  skip  datasets/il_tur/cjpe/expert/0000.parquet  (1.7MB, already present)
  skip  datasets/il_tur/cjpe/multi_dev/0000.parquet  (10.3MB, already present)
  skip  datasets/il_tur/cjpe/multi_train/0000.parquet  (223.4MB, already present)
  skip  datasets/il_tur/cjpe/multi_train/0001.parquet  (47.2MB, alread

In [13]:
d = DATA / "il_tur"
if not d.exists():
    print("IL-TUR not downloaded (WANT_ILTUR = False)")
else:
    grand = 0
    for cfg in sorted(p for p in d.iterdir() if p.is_dir()):
        parts = []
        for split in sorted(p for p in cfg.iterdir() if p.is_dir()):
            n = sum(len(pd.read_parquet(f)) for f in sorted(split.glob("*.parquet")))
            parts.append(f"{split.name}={n}")
            grand += n
        print(f"{cfg.name:6s} {', '.join(parts)}")
    print(f"\ntotal rows: {grand}")

bail   dev_all=17707, dev_specific=15929, test_all=35400, test_specific=36579, train_all=123742, train_specific=124341
cjpe   expert=56, multi_dev=994, multi_train=32305, single_dev=2511, single_train=5082, test=1517
lmt    acts=4036, cci_faq=1460, ip=1020
lner   fold_1=35, fold_2=35, fold_3=35


lsi    dev=10181, statutes=100, test=13019, train=42750
pcr    dev_candidates=1023, dev_queries=118, test_candidates=1727, test_queries=237, train_candidates=4320, train_queries=827
rr     CL_dev=5, CL_test=5, CL_train=40, IT_dev=5, IT_test=5, IT_train=40


summ   test=100, train=7030

total rows: 484316


## 8 · Manifest + final layout

Writes `datasets/MANIFEST.json` with size, sha256, and row count for every file.

In [14]:
def count_rows(p):
    try:
        if p.suffix == ".jsonl":
            return sum(1 for line in open(p, encoding="utf-8-sig") if line.strip())
        if p.suffix == ".json":
            obj = json.loads(p.read_text())
            return len(obj) if isinstance(obj, (list, dict)) else None
        if p.suffix == ".csv":
            return len(pd.read_csv(p))
        if p.suffix == ".parquet":
            return len(pd.read_parquet(p))
    except Exception:
        return None
    return None


manifest, total = {}, 0
for p in sorted(DATA.rglob("*")):
    if not p.is_file() or p.name in ("MANIFEST.json",):
        continue
    rel = str(p.relative_to(DATA))
    sz = p.stat().st_size
    total += sz
    manifest[rel] = {"bytes": sz, "sha256": sha256(p), "rows": count_rows(p)}

(DATA / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))

print(f"{'file':58s} {'size':>9s} {'rows':>8s}")
print("-" * 78)
for k, v in manifest.items():
    print(f"{k:58s} {human(v['bytes']):>9s} {str(v['rows'] if v['rows'] is not None else '-'):>8s}")
print("-" * 78)
print(f"{'TOTAL':58s} {human(total):>9s}")
print(f"\nmanifest -> {(DATA / 'MANIFEST.json').relative_to(ROOT)}")

file                                                            size     rows
------------------------------------------------------------------------------
README.md                                                      8.0KB        -
bns_bnss_bsa/bns_bnss_bsa_combined_clean.jsonl                 3.3MB     6354
bns_bnss_bsa/bns_qa.jsonl                                      1.1MB     2148
bns_bnss_bsa/bnss_qa.jsonl                                     1.7MB     3186
bns_bnss_bsa/bsa_qa.jsonl                                    504.2KB     1020
courtnav_rag/courtnav_rag_test.csv                            66.2KB       21
courtnav_rag/source_docs/doc1_special_power_of_attorney.pdf    44.1KB        -
courtnav_rag/source_docs/doc2_indian_contract_act.pdf        554.8KB        -
courtnav_rag/source_docs/doc3_drt_application.pdf            236.4KB        -
courtnav_rag/source_docs/doc4_civil_revision_petition.pdf    118.9KB        -
il_tur/bail/dev_all/0000.parquet                              

In [15]:
for p in sorted(DATA.rglob("*")):
    depth = len(p.relative_to(DATA).parts) - 1
    tag = "/" if p.is_dir() else ""
    print("  " * depth + "|- " + p.name + tag)

|- MANIFEST.json
|- README.md
|- bns_bnss_bsa/
  |- bns_bnss_bsa_combined_clean.jsonl
  |- bns_qa.jsonl
  |- bnss_qa.jsonl
  |- bsa_qa.jsonl
|- courtnav_rag/
  |- courtnav_rag_test.csv
  |- source_docs/
    |- doc1_special_power_of_attorney.pdf
    |- doc2_indian_contract_act.pdf
    |- doc3_drt_application.pdf
    |- doc4_civil_revision_petition.pdf
|- il_tur/
  |- bail/
    |- dev_all/
      |- 0000.parquet
    |- dev_specific/
      |- 0000.parquet
    |- test_all/
      |- 0000.parquet
    |- test_specific/
      |- 0000.parquet
    |- train_all/
      |- 0000.parquet
      |- 0001.parquet
    |- train_specific/
      |- 0000.parquet
      |- 0001.parquet
  |- cjpe/
    |- expert/
      |- 0000.parquet
    |- multi_dev/
      |- 0000.parquet
    |- multi_train/
      |- 0000.parquet
      |- 0001.parquet
    |- single_dev/
      |- 0000.parquet
    |- single_train/
      |- 0000.parquet
    |- test/
      |- 0000.parquet
  |- lmt/
    |- acts/
      |- 0000.parquet
    |- cci_faq/
